# Serverless LLMs and Agentic AI with Modal – Lesson 6
## Modal Classes & `@enter()` — Loading Models Once Per Container

In Lesson 4 you ran an Embedding Microservice on GPU. But there was a hidden inefficiency: **every time a container started, the model had to be re-loaded into memory**. For real LLMs (multi-GB weights), that cold-start cost is huge if you pay it on every call.

In this lesson you'll learn the canonical Modal pattern for AI serving:

**`@app.cls` + `@modal.enter()`** — load the model **once per container**, then serve **many** fast requests from that same warm process.

You'll also see Modal's full lifecycle hooks:

- **`@modal.enter()`** — runs once when the container starts (model load happens here)
- **`@modal.method()`** — runs on every call (cheap, the model is already in memory)
- **`@modal.exit()`** — runs once when the container is being shut down (cleanup)

### What you'll create

1. `TextGenerator` — a Modal Class that loads `distilgpt2` in `@modal.enter()` and exposes a `.generate()` method.
2. A demo that measures **cold-start vs warm-call** latency.
3. A fan-out demo with `.map()` so you can see multiple warm containers reused in parallel.

By the end you'll understand exactly **why classes (not plain functions) are the right tool for LLM inference on Modal** — and the mental model carries over to bigger models (Llama, Mistral, embedding models, rerankers, etc.).


In [ ]:
# =====================================
# Step 0 – Install and check Modal
# =====================================
!pip install modal --quiet
!which modal
!modal --version
print("✅ Modal installed.")

## Step 1 – Verify authentication


In [ ]:
# ============================
# Step 1 – Configure Modal using the CLI (matches docs)
# ============================
# ⚠️ IMPORTANT:
# - Replace the placeholder strings with your real MODAL_TOKEN_ID and MODAL_TOKEN_SECRET.
# - Do NOT commit these values to GitHub or share them.

TOKEN_ID = ""        # <-- paste from Modal dashboard
TOKEN_SECRET = ""    # <-- paste from Modal dashboard


if "YOUR_TOKEN_ID_HERE" in TOKEN_ID or "YOUR_TOKEN_SECRET_HERE" in TOKEN_SECRET:
    raise ValueError("❌ Please set TOKEN_ID and TOKEN_SECRET before running this cell.")

!modal token set --token-id $TOKEN_ID --token-secret $TOKEN_SECRET

print("✅ Token stored via `modal token set`. You should be authenticated now.")

Verifying token against https://api.modal.com
Token verified successfully!
⠋ Storing token
Token written to /root/.modal.toml in profile farhad-rh.
✅ Token stored via `modal token set`. You should be authenticated now.


## Step 2 – Write the Lesson 6 app

We use a small open model (`distilgpt2`, ~330 MB) so the lesson runs quickly even on a free GPU.
The same exact pattern works for Llama, Mistral, embedding models, rerankers, etc. — only the model name changes.

Two important design choices in this app:

1. **Pre-bake the model into the image** with `image.run_function(_download_model)`.
   This way the weights are already on disk when a container starts — `@enter` just loads from local cache, no network download.

2. **Use `@app.cls` + `@modal.enter`** instead of a plain `@app.function`.
   The model is loaded once per container and reused across all `.generate()` calls.

### Why not a plain function?
A plain `@app.function` *can* keep its container warm too, but module-level model loading runs in awkward places, makes testing harder, and you don't get clean `@enter`/`@exit` hooks. Classes are the idiomatic Modal pattern for any stateful workload — especially AI inference.


In [ ]:
%%writefile lesson6_classes_llm.py
import time
from typing import List, Dict

import modal

# ------------------------------------------------------------
# Image: PyTorch + transformers, with the model baked in at build time
# ------------------------------------------------------------
MODEL_NAME = "distilgpt2"


def _download_model():
    """Runs at IMAGE BUILD time so weights are cached on the container's disk."""
    from transformers import AutoModelForCausalLM, AutoTokenizer
    AutoTokenizer.from_pretrained(MODEL_NAME)
    AutoModelForCausalLM.from_pretrained(MODEL_NAME)


image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install("torch==2.3.0", "transformers==4.44.2")
    .run_function(_download_model)
)

app = modal.App("lesson6-classes-llm", image=image)


# ------------------------------------------------------------
# Modal Class: load model once per container, serve many calls
# ------------------------------------------------------------
@app.cls(gpu="T4", scaledown_window=60)
class TextGenerator:
    """One instance per container.

    - @modal.enter()  : runs ONCE on container start  -> load model
    - @modal.method() : runs on EVERY call            -> inference (fast)
    - @modal.exit()   : runs ONCE on container shutdown -> cleanup
    """

    @modal.enter()
    def load(self):
        print(f"[enter] loading {MODEL_NAME} ...")
        t0 = time.time()
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        self.model = (
            AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(self.device)
        )
        self.model.eval()
        print(f"[enter] model ready on {self.device} in {time.time()-t0:.2f}s")

    @modal.exit()
    def cleanup(self):
        print("[exit] container shutting down")

    @modal.method()
    def generate(self, prompt: str, max_new_tokens: int = 32) -> Dict:
        """Run inference. Model is already loaded -> cheap."""
        import torch
        t0 = time.time()
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        text = self.tokenizer.decode(out[0], skip_special_tokens=True)
        return {
            "prompt": prompt,
            "completion": text,
            "device": self.device,
            "latency_s": round(time.time() - t0, 3),
        }


@app.local_entrypoint()
def lesson6_main():
    """Demonstration sequence.

    Shows the cold-start cost (call 1) vs warm-call cost (call 2+),
    then fans out across many prompts in parallel.
    """
    print("\n=========================================")
    print("Lesson 6 – Modal Classes & @enter()")
    print("=========================================\n")

    gen = TextGenerator()

    # ---- Call 1: COLD start ----
    # total time = container boot + @enter (model load) + inference
    t0 = time.time()
    r1 = gen.generate.remote("Once upon a time, in a serverless cloud,")
    print(f"\n[Call 1 – cold]   total={time.time()-t0:.2f}s   inference={r1['latency_s']}s")
    print(" ->", r1["completion"][:120], "...")

    # ---- Call 2: WARM ----
    # total time = inference only (container already up, model already loaded)
    t0 = time.time()
    r2 = gen.generate.remote("The agent decided that the next step was")
    print(f"\n[Call 2 – warm]   total={time.time()-t0:.2f}s   inference={r2['latency_s']}s")
    print(" ->", r2["completion"][:120], "...")

    # ---- Fan-out: many prompts in parallel via .map() ----
    # Modal will reuse warm containers and spin up new ones if needed.
    prompts: List[str] = [
        "Tool use means",
        "A retrieval augmented system",
        "The embedding vector represents",
        "GPU acceleration is useful when",
        "The planner agent receives",
        "Serverless inference scales by",
        "A short story about a robot:",
        "Top three reasons to use Modal:",
    ]
    print(f"\n[Fan-out] generating {len(prompts)} prompts in parallel via .map() ...")
    t0 = time.time()
    results = list(gen.generate.map(prompts))
    print(f"[Fan-out] {len(results)} completions in {time.time()-t0:.2f}s\n")
    for r in results:
        print(f"  {r['latency_s']}s | {r['prompt']!r:40s} -> {r['completion'][:60]}...")

    print("\n✅ Open the Modal dashboard → Apps → lesson6-classes-llm to see containers warm-reused.")


## Step 3 – Run the demo

This will:
- build the image (first time only — installs torch + transformers, downloads model)
- start a GPU container and load the model in `@enter`
- run a cold call, then a warm call, then a fan-out of 8 prompts

**Watch the numbers carefully:** call 1 includes the cold-start cost, call 2 should be much faster.


In [ ]:
!modal run lesson6_classes_llm.py

## Step 4 – Experiment: keep containers warm

Cold starts are expensive for LLMs because of the model-load step in `@enter`. Modal lets you keep one or more containers always warm so that the *next* user request never pays the cold-start tax.

Two knobs you should know:

- **`min_containers=N`** — keep at least N containers warm at all times. Great for low-latency LLM APIs.
- **`scaledown_window=SECONDS`** — how long an idle container sticks around before being killed.

Try editing the class decorator and re-running:

```python
@app.cls(gpu="T4", min_containers=1, scaledown_window=300)
class TextGenerator:
    ...
```

Now run the demo again — the **first call after the change** will still be cold (new container provisioned), but every subsequent run of the demo should start with a warm container and skip the model-load delay.

> ⚠️ `min_containers > 0` means you are paying for an always-on GPU. Set it back to 0 (or remove the kwarg) when you're done experimenting.


## Step 5 – Inspect & debug from the CLI

Just like in Lesson 5, you can list and shell into your app for debugging.


In [ ]:
!modal app list

In [ ]:
!modal shell lesson6_classes_llm.py::TextGenerator.generate

## Recap & what's next

You learned:

- **Why Modal Classes exist**: stateful workloads (especially AI inference) need a place to put expensive setup like model loading.
- **`@modal.enter()` is the canonical “load the model here” hook** — runs once per container.
- **`@modal.method()` is the per-request hook** — cheap because the model is already in memory.
- **`@modal.exit()`** lets you do cleanup before a container is killed.
- **`min_containers` / `scaledown_window`** trade money for latency by keeping containers warm.
- **Classes compose with `.map()`** just like functions — so you can fan out across many warm GPU containers easily.